# Models & Ticket Schema

> Data models, lifecycle statuses, and JSON serialization for `solveitcrafts` support tickets and review manifests.

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import json, secrets
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any, Literal
from fastcore.utils import *

TicketStatus = Literal["new", "triaged", "in_review", "merged", "closed"]
VALID_STATUSES = ("new", "triaged", "in_review", "merged", "closed")

## Timestamps & Ticket Identifiers

Ticket IDs follow the compact, sortable format `TKT-YYYYMMDD-XXXXXX` (UTC date plus 6 uppercase hex characters from `secrets.token_hex(3)`), and timestamps are serialized as second-precision UTC ISO-8601 strings ending in `Z`.

In [ ]:
#| export
def utc_now_iso() -> str:
    "Return the current UTC timestamp as a second-precision ISO-8601 string."
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def new_ticket_id(
    prefix: str = "TKT", # Ticket ID prefix (e.g. `TKT` or `BUG`)
) -> str:
    "Generate a unique, date-prefixed ticket identifier."
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
    return f"{prefix}-{stamp}-{secrets.token_hex(3).upper()}"

In [ ]:
show_doc(utc_now_iso)
show_doc(new_ticket_id)

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
ts = utc_now_iso()
assert ts.endswith("Z") and "T" in ts
parts = new_ticket_id().split("-")
test_eq((parts[0], len(parts[1]), len(parts[2])), ("TKT", 8, 6))
assert new_ticket_id("BUG").startswith("BUG-")

## Capture Metadata, Triage Notes & Drive References

Each ticket records:
- `CaptureMeta`: browser URL, viewport dimensions, user agent, recent console errors, and optional Solveit `dialog_id`.
- `TriageNote`: timestamped operator notes appended during triage.
- `DriveRefs`: Google Drive file/folder IDs and web links for `ticket.json`, sanitized `dom.html`, and optional `screenshot.png`.

In [ ]:
#| export
_DRIVE_REF_KEYS = ("folder_id", "folder_url", "ticket_file_id", "dom_file_id", "dom_url", "screenshot_file_id", "screenshot_url")

@dataclass
class CaptureMeta:
    "Browser environment metadata attached to a bug capture."
    url: str = ""
    viewport: dict[str, int] = field(default_factory=lambda: dict(width=0, height=0))
    user_agent: str = ""
    console_errors: list[str] = field(default_factory=list)
    captured_at: str = field(default_factory=utc_now_iso)
    dialog_id: str | None = None

    @classmethod
    def from_dict(cls, data: dict[str, Any] | None) -> CaptureMeta:
        if not data: return cls()
        vp = data.get("viewport") or dict(width=0, height=0)
        return cls(url=str(data.get('url', '')), viewport=dict(width=int(vp.get('width', 0)), height=int(vp.get('height', 0))),
            user_agent=str(data.get('user_agent', '')), console_errors=[str(e) for e in data.get('console_errors') or []],
            captured_at=str(data.get('captured_at') or utc_now_iso()), dialog_id=data.get('dialog_id'))

@dataclass
class TriageNote:
    "Operator triage note stored inside `ticket.json`."
    author: str
    text: str
    created_at: str = field(default_factory=utc_now_iso)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> TriageNote:
        return cls(author=str(data.get('author', 'operator')), text=str(data.get('text', '')),
            created_at=str(data.get('created_at') or utc_now_iso()))

@dataclass
class DriveRefs:
    "Google Drive (or local mirror) identifiers and links for a ticket bundle."
    folder_id: str | None = None
    folder_url: str | None = None
    ticket_file_id: str | None = None
    dom_file_id: str | None = None
    dom_url: str | None = None
    screenshot_file_id: str | None = None
    screenshot_url: str | None = None

    @classmethod
    def from_dict(cls, data: dict[str, Any] | None) -> DriveRefs:
        if not data: return cls()
        return cls(**{k: data.get(k) for k in _DRIVE_REF_KEYS})

In [ ]:
cap = CaptureMeta.from_dict(dict(url="https://solve.it.com/d/1", viewport=dict(width="1280", height="720"), console_errors=["err1"]))
test_eq(cap.viewport, dict(width=1280, height=720))
test_eq(cap.console_errors, ["err1"])
test_eq(CaptureMeta.from_dict(None).url, "")
assert DriveRefs.from_dict(None).folder_id is None
note = TriageNote.from_dict(dict(text="Needs repro"))
test_eq((note.author, note.text), ("operator", "Needs repro"))

## `Ticket` Lifecycle & Serialization

A `Ticket` moves through the validated lifecycle `new` → `triaged` → `in_review` → `merged` | `closed`:
- Adding a triage note to a `new` ticket automatically promotes it to `triaged`.
- Invalid status strings raise `ValueError` in both `__post_init__` and `set_status`.
- `index_entry()` produces the lightweight summary written to `solveitcrafts/index.json` so listing and filtering by status does not require scanning every folder.

In [ ]:
#| export
def _validate_status(status: str) -> None:
    if status not in VALID_STATUSES: raise ValueError(f"Invalid ticket status {status!r}; expected one of {VALID_STATUSES}")

@dataclass
class Ticket:
    "Canonical support ticket persisted as `ticket.json` in Drive or local storage."
    id: str
    title: str
    description: str
    reporter_notes: str = ""
    status: TicketStatus = "new"
    created_at: str = field(default_factory=utc_now_iso)
    updated_at: str = field(default_factory=utc_now_iso)
    capture: CaptureMeta = field(default_factory=CaptureMeta)
    triage_notes: list[TriageNote] = field(default_factory=list)
    branch: str | None = None
    base_branch: str | None = None
    pr_number: int | None = None
    pr_url: str | None = None
    review_app_id: str | None = None
    review_app_url: str | None = None
    group_id: str | None = None
    drive: DriveRefs = field(default_factory=DriveRefs)

    def __post_init__(self) -> None: _validate_status(self.status)
    def touch(self) -> None: self.updated_at = utc_now_iso()

    def set_status(
        self,
        status: TicketStatus, # New validated lifecycle status
    ) -> None:
        "Transition the ticket to a validated status and bump `updated_at`."
        _validate_status(status)
        self.status = status
        self.touch()

    def add_note(
        self,
        text: str, # Triage note text (stripped of surrounding whitespace)
        author: str = "operator", # Note author label
    ) -> TriageNote:
        "Append a triage note and promote `new` tickets to `triaged`."
        note = TriageNote(author=author, text=text.strip())
        self.triage_notes.append(note)
        if self.status == "new": self.status = "triaged"
        self.touch()
        return note

    def to_dict(self) -> dict[str, Any]: return asdict(self)
    def to_json(self, indent: int = 2) -> str: return json.dumps(self.to_dict(), indent=indent, ensure_ascii=False)

    def index_entry(self) -> dict[str, Any]:
        return dict(id=self.id, title=self.title, status=self.status, created_at=self.created_at, updated_at=self.updated_at,
            branch=self.branch, base_branch=self.base_branch, pr_number=self.pr_number, pr_url=self.pr_url,
            review_app_id=self.review_app_id, review_app_url=self.review_app_url, group_id=self.group_id, folder_id=self.drive.folder_id,
            folder_url=self.drive.folder_url)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> Ticket:
        return cls(
            id=str(data["id"]), title=str(data.get("title", "")), description=str(data.get("description", "")),
            reporter_notes=str(data.get("reporter_notes", "")), status=data.get("status", "new"),
            created_at=str(data.get("created_at") or utc_now_iso()), updated_at=str(data.get("updated_at") or utc_now_iso()),
            capture=CaptureMeta.from_dict(data.get("capture")),
            triage_notes=[TriageNote.from_dict(n) for n in (data.get("triage_notes") or [])],
            branch=data.get("branch"), base_branch=data.get("base_branch"),
            pr_number=data.get("pr_number"), pr_url=data.get("pr_url"),
            review_app_id=data.get("review_app_id"), review_app_url=data.get("review_app_url"),
            group_id=data.get("group_id"), drive=DriveRefs.from_dict(data.get("drive")))

    @classmethod
    def from_json(cls, raw: str | bytes) -> Ticket:
        text = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else raw
        return cls.from_dict(loads(text))

In [ ]:
show_doc(Ticket)

Adding a triage note promotes a `new` ticket to `triaged` and roundtrips cleanly through JSON text and UTF-8 bytes.

In [ ]:
t = Ticket(id='TKT-20261123-ABC123', title='Broken table sort', description='Numeric column sorts alphabetically',
    capture=CaptureMeta(url='https://solve.it.com/d/1', viewport=dict(width=1440, height=900)))
test_eq(t.status, "new")
t.add_note("  Reproduced on Chrome 130  ")
test_eq((t.status, t.triage_notes[0].text), ("triaged", "Reproduced on Chrome 130"))

rebuilt = Ticket.from_json(t.to_json())
rebuilt_bytes = Ticket.from_json(t.to_json().encode("utf-8"))
test_eq((rebuilt.id, rebuilt_bytes.id, rebuilt.status, rebuilt.capture.viewport["width"]), (t.id, t.id, "triaged", 1440))
idx = t.index_entry()
test_eq((idx["id"], idx["status"]), (t.id, "triaged"))

Invalid status values are rejected both on construction and during `set_status`.

In [ ]:
test_fail(lambda: Ticket(id="TKT-BAD", title="Bad", description="Bad", status="invalid"), ValueError)
test_fail(lambda: t.set_status("bogus"), ValueError)

## `ReviewManifest`

When one or more tickets are assigned to a branch, PR, and Solveit review app, `ReviewManifest` packages their metadata and Drive file IDs into `review_manifest.json` so an agent running inside the review app can pull the sanitized DOM and screenshots via `solvedrive`.

In [ ]:
#| export
@dataclass
class ReviewManifest:
    "Context bundle handed off to a Solveit review app so an agent can inspect tickets via solvedrive."
    group_id: str
    branch: str
    base_branch: str
    pr_number: int | None = None
    pr_url: str | None = None
    review_app_id: str | None = None
    review_app_url: str | None = None
    created_at: str = field(default_factory=utc_now_iso)
    tickets: list[dict[str, Any]] = field(default_factory=list)

    @classmethod
    def from_tickets(
        cls,
        tickets: list[Ticket], # Tickets included in this review bundle
        *,
        group_id: str, # Shared review group identifier
        branch: str, # Git branch backing the review app
        base_branch: str, # Base branch for the PR (e.g. `main` or stacked branch)
        pr_number: int | None = None, # GitHub pull request number
        pr_url: str | None = None, # GitHub pull request URL
        review_app_id: str | None = None, # Solveit review-app instance name
        review_app_url: str | None = None, # Solveit review-app live URL
    ) -> ReviewManifest:
        entries = [dict(id=t.id, title=t.title, description=t.description, reporter_notes=t.reporter_notes, status=t.status,
            capture=asdict(t.capture), triage_notes=[asdict(n) for n in t.triage_notes], drive=asdict(t.drive)) for t in tickets]
        return cls(group_id=group_id, branch=branch, base_branch=base_branch, pr_number=pr_number, pr_url=pr_url,
            review_app_id=review_app_id, review_app_url=review_app_url, tickets=entries)

    def to_dict(self) -> dict[str, Any]: return asdict(self)
    def to_json(self, indent: int = 2) -> str: return json.dumps(self.to_dict(), indent=indent, ensure_ascii=False)

In [ ]:
show_doc(ReviewManifest)

In [ ]:
manifest = ReviewManifest.from_tickets([t], group_id='GRP-001', branch='support/tkt-sort', base_branch='main', pr_number=42,
    pr_url='https://github.com/abiancar/solve-it-crafts/pull/42')
parsed = loads(manifest.to_json())
test_eq((parsed["group_id"], parsed["pr_number"], len(parsed["tickets"]), parsed["tickets"][0]["id"]), ("GRP-001", 42, 1, t.id))